<a href="https://colab.research.google.com/github/ChristinaCarvalho/Meter-Data-Prediction-AI-Project/blob/main/Meter_AI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# resource monitoring
!free -h
!uptime

In [2]:
# Manually disables runtime
from google.colab import runtime
print('Do you want to unassign the runtime? (y/n)')
user_input = input()
if user_input == 'y':
    print(runtime.unassign())
else:
    print('Runtime still active.')

In [ ]:
# Mount drive
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

filename = '/content/drive/MyDrive/AI_Meter_Project/Meter_data_test_for_Export.csv'

# Creates a pandas dataframe from CSV file
df = pd.read_csv(filename)
df = df.drop(df.columns[[4,5,6,7,8,9,10,11,12,13,14,15,16,17]], axis=1)


#df['CollectDts'] = pd.to_datetime(df['CollectDts'], format='mixed').dt.round('1min') rounds to closest min
df['CollectDts'] = pd.to_datetime(df['CollectDts'], format='mixed').dt.round('1min') #floors to minute
df = df.set_index('CollectDts')
df = df.resample('1min').mean()

#Print sample of data
#print(df.to_string())

#Configuring data as a tensor
training_data = torch.tensor(df.values)
print(training_data)
height, width = training_data.shape
print(f'Tensor Height: {height}, Width:{width}')

#Testing sliding window
window_height = 35
stride = 5
padded_data = torch.nn.functional.pad(training_data, (0, 0, window_height - 1, 0))
unfolded_data = training_data.unfold(0, window_height, stride)
print("2D sliding windows shape:", unfolded_data.shape)




In [ ]:
#Test model from https://www.codegenes.net/blog/bi-directional-lstm-attention-pytorch/

class BiLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_prob):
        super(BiLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(input_size, hidden_size, bidirectional=True)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc = nn.Linear(hidden_size * 2, output_size)
        self.attn = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_weights = torch.softmax(self.attn(lstm_out), dim=0)
        context = torch.sum(attn_weights * lstm_out, dim=0)
        output = self.fc(context)
        return output

input_size = 1
hidden_size = 64
num_layers = 2
num_classes = 2
dropout_prob = 0.2
model_with_dropout = BiLSTM(input_size, hidden_size, num_classes, dropout_prob)

In [ ]:
#Test model from https://www.codegenes.net/blog/bi-directional-lstm-attention-pytorch/

# Hyperparameters
input_size = 10
hidden_size = 20
output_size = 2
batch_size = 32
seq_len = 5

# Initialize the model
model = BiLSTM(input_size, hidden_size, output_size)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Generate some random data for demonstration
x = torch.randn(seq_len, batch_size, input_size)
y = torch.randint(0, output_size, (batch_size,))

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(x)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    print(f'Epoch {epoch + 1}/{num_epochs}, Loss: {loss.item()}')